In [1]:
from msi.msi import MSI
from diff_prof.diffusion_profiles import DiffusionProfiles
import multiprocessing
import numpy as np
import pickle
import networkx as nx
import polars as pl
import gc
import seaborn as sns
import matplotlib.pyplot as plt
#import paramiko

from filter_logic import filter_drug_protein_edges
from tests.msi import test_msi
from tests.diff_prof import test_diffusion_profiles

In [2]:
DRUG_COL = 'drugbank_id'
TGT_COL = 'uniprot_id'
LABEL_COL = 'sample_type'  # positive / negative_balanced
PAIR_COL = 'compound_target_pair'

In [ ]:
df_lazy = pl.scan_parquet("data/combined_filtered_annotated_docking_results.parquet")

In [ ]:
def _pair_best_score(df_in: pl.DataFrame, score_col: str, best: str) -> pl.DataFrame:
    if score_col not in df_in.columns:
        print(f"Warning: column not found, skipping: {score_col}")
        return pl.DataFrame()

    if best not in {"min", "max"}:
        raise ValueError("best must be 'min' or 'max'")

    agg_score = (pl.min(score_col) if best == "min" else pl.max(score_col)).alias(score_col)

    return (
        df_in
        .filter(pl.col(score_col).is_not_null() & pl.col(score_col).is_finite())
        .group_by(PAIR_COL)
        .agg([
            pl.first(DRUG_COL).alias(DRUG_COL),
            pl.first(TGT_COL).alias(TGT_COL),
            pl.first(LABEL_COL).alias(LABEL_COL),
            agg_score,
        ])
    )

In [ ]:
df_gnina  = _pair_best_score(df_lazy, "GNINA_Score",  best="min").collect()
df_smina  = _pair_best_score(df_lazy, "SMINA_Score",  best="min").collect()
df_ledock = _pair_best_score(df_lazy, "Ledock_Score", best="min").collect()
df_gold   = _pair_best_score(df_lazy, "GOLD_Score",   best="max").collect()

In [ ]:
df_DeepDTA = _pair_best_score(df_lazy, "dta_DeepDTA_ic50", best="max").collect()
df_GAT_GCN = _pair_best_score(df_lazy, "dta_GAT_GCN_ic50", best="max").collect()
df_GCNNet = _pair_best_score(df_lazy, "dta_GCNNet_ic50", best="max").collect()
df_GINConvNet = _pair_best_score(df_lazy, "dta_GINConvNet_ic50", best="max").collect()
df_MLTLE_GCN = _pair_best_score(df_lazy, "dta_MLTLE_GCN_ic50", best="max").collect()
df_MLTLE_GIN = _pair_best_score(df_lazy, "dta_MLTLE_GIN_ic50", best="max").collect()

In [ ]:
tools_scores_dfs = {"GNINA": df_gnina, "SMINA": df_smina, "LEDOCK": df_ledock, "GOLD": df_gold,
                    "DeepDTA": df_DeepDTA, "GAT_GCN": df_GAT_GCN, "GCNNet": df_GCNNet,
                    "GINConvNet": df_GINConvNet, "MLTLE_GCN": df_MLTLE_GCN, "MLTLE_GIN": df_MLTLE_GIN}

In [ ]:
# Load your dataframes efficiently
df_edges = pl.read_csv("data/1_drug_to_protein.tsv", separator="\t")
df_mapping = pl.read_csv("data/uniprot_gene_mapping.csv")

# Run the filter on all docking-based scores as well as ML-based scores
# by iterating over the tools_scores_dfs dictionary
cutoff_ranges = {}

# Use a range between -7.0 and -9.0 for docking scores (SMINA, GNINA)
# Use a range between -5.0 and -7.0 for LEDOCK
# Use a range between 60 and 90 for GOLD
# Use a range between 5 and 8 for ML-based scores
cutoff_ranges['GNINA'] = np.arange(-7.0, -9.1, -0.5)
cutoff_ranges['SMINA'] = np.arange(-7.0, -9.1, -0.5)
cutoff_ranges['LEDOCK'] = np.arange(-5.0, -7.1, -0.5)
cutoff_ranges['GOLD'] = np.arange(60, 91, 10)
cutoff_ranges['DeepDTA'] = np.arange(5, 9, 1)
cutoff_ranges['GAT_GCN'] = np.arange(5, 9, 1)
cutoff_ranges['GCNNet'] = np.arange(5, 9, 1)
cutoff_ranges['GINConvNet'] = np.arange(5, 9, 1)
cutoff_ranges['MLTLE_GCN'] = np.arange(5, 9, 1)
cutoff_ranges['MLTLE_GIN'] = np.arange(5, 9, 1) 

tool_score_columns = {
    "GNINA": "GNINA_Score",
    "SMINA": "SMINA_Score",
    "LEDOCK": "Ledock_Score",
    "GOLD": "GOLD_Score",
    "DeepDTA": "dta_DeepDTA_ic50",
    "GAT_GCN": "dta_GAT_GCN_ic50",
    "GCNNet": "dta_GCNNet_ic50",
    "GINConvNet": "dta_GINConvNet_ic50",
    "MLTLE_GCN": "dta_MLTLE_GCN_ic50",
    "MLTLE_GIN": "dta_MLTLE_GIN_ic50"
}

for tool_name, df_scores in tools_scores_dfs.items():
    if df_scores.is_empty():
        print(f"Skipping filtering for {tool_name} due to missing scores.")
        continue

    better = 'lower' if tool_name in {"GNINA", "SMINA", "LEDOCK"} else 'higher'
    for cutoff in cutoff_ranges[tool_name]:
        score_cutoff = cutoff
        filtered_edges = filter_drug_protein_edges(
            df_edges, 
            df_mapping, 
            df_scores, 
            score_column=tool_score_columns[tool_name], 
            better=better, 
            score_cutoff=score_cutoff,
            edge_source='from_scores',
            on_unmapped_protein='drop',
        )
        output_path = f"data/1_drug_to_protein_filtered_{tool_name.lower()}_cutoff_{score_cutoff}.tsv"
        filtered_edges.write_csv(output_path, separator="\t")
        print(f"Saved filtered edges for {tool_name} with cutoff {score_cutoff} to {output_path}")

In [ ]:
# Batch wrapper usage: build MSI + recompute ALL diffusion profiles
# Note: this can take a while and writes many .npy files.
from diff_prof import (
    compute_all_diffusion_profiles_for_msi,
    compute_all_diffusion_profiles_for_msi_across_filtered_drug2protein_tsvs,
    diffusion_profile_similarity,
 )

# Example A: default MSI files (same as MSI())
profiles_default, msi_default = compute_all_diffusion_profiles_for_msi(
    save_load_file_path="results/",
    num_cores=12,
 )

# Example B: compute diffusion profiles for EVERY filtered drug→protein TSV we generated
# One output directory per TSV under results_filtered/
runs_filtered = compute_all_diffusion_profiles_for_msi_across_filtered_drug2protein_tsvs(
    save_root="results_filtered/",
    drug2protein_glob="data/1_drug_to_protein_filtered_*cutoff_*.tsv",
    recompute=True,  # set to False to just load from disk if already computed
    on_error="skip",
    num_cores=12,
 )

print(f"Computed runs: {len(runs_filtered)}")
example_run_ids = sorted(runs_filtered.keys())[:10]
print("Example run_ids:", example_run_ids)

# Pick one run to use in downstream cells (keeps the rest of the notebook working)
selected_run_id = example_run_ids[0] if example_run_ids else None
if selected_run_id is None:
    raise RuntimeError("No filtered TSV runs found; check the glob or TSV generation cell")
print("Selected run:", selected_run_id)
profiles_gat = runs_filtered[selected_run_id].profiles
msi_gat = runs_filtered[selected_run_id].msi

In [ ]:
# Parallel similarity computation: default MSI vs filtered MSI (per drug)
import multiprocessing as mp
import numpy as np
import pandas as pd

# Restrict to drugs present in BOTH graphs AND both profile dicts
drugs_default = set(msi_default.drugs_in_graph)
drugs_gat = set(msi_gat.drugs_in_graph)
common_drugs = sorted((drugs_default & drugs_gat) & set(profiles_default.keys()) & set(profiles_gat.keys()))
print(f"Common drugs to compare: {len(common_drugs)}")

# Precompute intersection-node indices ONCE (shared by all drugs)
common_nodes = [n for n in msi_default.nodelist if n in msi_gat.node2idx]
if len(common_nodes) == 0:
    raise ValueError("No common nodes between MSI graphs; cannot compare profiles")

idx_a = np.fromiter((msi_default.node2idx[n] for n in common_nodes), dtype=np.int64)
idx_b = np.fromiter((msi_gat.node2idx[n] for n in common_nodes), dtype=np.int64)
n_common = int(len(common_nodes))
print(f"Common nodes for alignment: {n_common}")

# Use fork on Linux so the large profile dicts are shared copy-on-write
ctx = mp.get_context("fork")
num_workers = max(1, int(mp.cpu_count() * 0.6))
num_workers = min(num_workers, len(common_drugs) if len(common_drugs) > 0 else 1)
print(f"Using workers: {num_workers}")

# Globals for worker processes (inherited via fork)
_P_DEFAULT = profiles_default
_P_GAT = profiles_gat
_IDX_A = idx_a
_IDX_B = idx_b

def _similarity_one_drug(drug_id: str):
    a = _P_DEFAULT[drug_id][_IDX_A]
    b = _P_GAT[drug_id][_IDX_B]
    dist = float(np.linalg.norm(a - b, ord=2))
    sim = 1.0 / (1.0 + dist)
    norm_a = float(np.linalg.norm(a, ord=2))
    norm_b = float(np.linalg.norm(b, ord=2))
    rel_dist = dist / (norm_a + 1e-12)
    sum_a = float(np.sum(a))
    sum_b = float(np.sum(b))
    return (drug_id, sim, dist, rel_dist, norm_a, norm_b, sum_a, sum_b)

if len(common_drugs) == 0:
    df_similarity_default_vs_filtered = pd.DataFrame(
        columns=[
            "drug_id","similarity_l2","l2_distance","relative_l2_distance","l2_norm_default","l2_norm_filtered","sum_default","sum_filtered","n_common_nodes"
        ]
    )
else:
    with ctx.Pool(processes=num_workers) as pool:
        # chunksize tuned to reduce overhead for large drug lists
        chunksize = max(1, len(common_drugs) // (num_workers * 8) if num_workers else 1)
        rows = list(pool.imap_unordered(_similarity_one_drug, common_drugs, chunksize=chunksize))
    df_similarity_default_vs_filtered = pd.DataFrame(
        rows,
        columns=[
            "drug_id","similarity_l2","l2_distance","relative_l2_distance","l2_norm_default","l2_norm_filtered","sum_default","sum_filtered"
        ],
    )
    df_similarity_default_vs_filtered["n_common_nodes"] = n_common
    df_similarity_default_vs_filtered = df_similarity_default_vs_filtered.sort_values("similarity_l2", ascending=False).reset_index(drop=True)

df_similarity_default_vs_filtered.head()

In [ ]:
# Plot distributions of similarity_l2 in df_similarity_default_vs_filtered
sns.kdeplot(df_similarity_default_vs_filtered,x='similarity_l2')

In [ ]:
# ------------------------------------------------------------
# Example C: compare profiles returned by compute_all_diffusion_profiles_for_msi
# Same drug (DB01098) across two different MSI graphs -> intersection alignment
sim_wrap, diag_wrap = diffusion_profile_similarity(
    profiles_default["DB01098"],
    profiles_gat["DB01098"],
    method="l2",
    align="intersection",
    msi_a=msi_default,
    msi_b=msi_gat,
    normalization=None,
    return_diagnostics=True,
 )
print(f"[Wrapper] DB01098 default vs filtered: similarity={sim_wrap:.6g}, dist={diag_wrap['l2_distance']:.6g}, n_common={diag_wrap['n_common']}")

In [ ]:

# Two drugs within the SAME returned dict -> position alignment works
other_key_gat = next(k for k in profiles_gat.keys() if k != "DB01098")
sim_same, diag_same = diffusion_profile_similarity(
    profiles_gat["DB01098"],
    profiles_gat[other_key_gat],
    method="l2",
    align="position",
    return_diagnostics=True,
 )
print(f"[Wrapper] DB01098 vs {other_key_gat}: similarity={sim_same:.6g}, dist={diag_same['l2_distance']:.6g}")

### Reserved code (dumplings? :) )

In [ ]:
# Just some template code to demonstrate connection to remote server and read a parquet file
host = "headscale.bio-cloud.site"
username = "onur"
port = 50002
# password = "your_password_here"  # or use key-based authentication

ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
ssh.connect(host, username=username, password=password, port=port)  # add password/key args as needed
sftp = ssh.open_sftp()
remote_file = sftp.open("/media/onur/Elements/cavity_space_consensus_docking/2025_06_29_batch_dock/dta_atlas_data/home/madinasu/convertedData/dta_atlas_dataset_1_0/part.0.parquet", "rb")
df = pl.read_parquet(remote_file)
remote_file.close()
sftp.close()
ssh.close()